# Reproduce the test predictions

This notebook rebuilds our submission for the Solar Filament Segmentation Challenge 2026 from the test JPEG
files and the trained weights stored in this repository (`weights/`). It runs the pipeline of
`src/solarfil/pipeline.py` on every test frame and writes `submission.csv` in the leaderboard format.

Steps, for each frame: flat field (`preprocess.py`); the dense map, the mean of two U-Nets trained on all labelled
frames (seed 0 of the recipe and network W, whose map, trained on targets moved down by 0.24 px, moves back up), with
four flips each (`infer.py`); the component tree with watershed split rows (`tree.py`); the detections of two Mask
R-CNN instance models on the flat field at 1024 px (`instances.py`: one trained on one annotator record per sample on
folds that hold out whole annotation groups, one trained on every filament that any annotator of the frame drew);
node descriptors with each detector's six columns and the scorer (`bank.py`, `scorer.py`); the exact selection at the
fixed threshold tau (`tree.select`); masks.
The design and the out-of-fold validation are in `DESIGN.md`.

## 1. Settings

Set `DATA` to the official competition folder and keep `WEIGHTS` at the contained `weights/` folder.
Only the JPEGs in `test/test_images` are read. `DEVICE = "cpu"` and `TAU = None` select the documented
CPU reference at cutoff 0.30 (public PQ 0.41). Set `DEVICE = "cuda"` to use a CUDA GPU, or `"mps"`
for Apple GPU execution. Backend precision can change masks; the references are named by backend.
`TORCH_THREADS` defaults to 4. `TAU = 0.26` selects the other retained operating point.

`LIMIT = None` processes all 180 test frames. A number limits the run to a quick sample.
`OUTLINE = False` keeps the submitted outline rule. The optional growth step uses `T_LOW`.


In [ ]:
from pathlib import Path

DATA = Path("data/MAGFiLO_1.0_Kaggle_2026")         # the competition data
WEIGHTS = Path("weights")                           # trained weights stored in this repository
DEVICE = "cpu"                                     # cpu: scored reference; cuda: faster GPU inference
TORCH_THREADS = 4                                  # CPU threads used by PyTorch
TAU = None                                          # None: the weights folder's tau (0.30)
OUTLINE = False                                     # True with T_LOW = 0.3: our final selection (58281141)
T_LOW = 0.1                                         # the outline step's map level (morph.grow_thin's t_low)
LIMIT = None                                        # a number: only the first LIMIT test frames (a quick check)
OUT = Path("submission.csv")

## 2. Environment

Use Python 3.12 with the pinned `uv.lock` environment. From this repository, run
`uv run --group notebook jupyter lab` and open this notebook. The next cell imports `src/solarfil`
and prints the environment and model manifest. Trained weights load from the repository.

CPU and MPS use float32; CUDA uses bfloat16 for the dense maps. The CPU reference has 1,152 masks
at cutoff 0.30. Complete CUDA demonstrations run faster and can produce a different mask file.
`README.md` and `VERIFICATION.md` give the measured execution and comparison scopes.


In [ ]:
import sys

sys.path.insert(0, str(Path("src").resolve()))  # the solarfil package of this repository
import torch

torch.set_num_threads(TORCH_THREADS)

print("torch", torch.__version__, "| CUDA GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print((WEIGHTS / "manifest.json").read_text())

## 3. Predict every test frame

In [ ]:
import time

from solarfil import submission
from solarfil.pipeline import Pipeline

pipe = Pipeline.load(WEIGHTS, dev=torch.device(DEVICE))
if TAU is not None:
    pipe.tau = TAU
if OUTLINE:
    pipe.outline = {"t_low": T_LOW}  # the thin-protrusion outline step (morph.grow_thin), on only in the outline0.3 reference CSV
device = torch.cuda.get_device_name(0) if pipe.dev.type == "cuda" else pipe.dev.type
print(f"Using {device}; device precision can change masks (VERIFICATION.md).")
paths = sorted((DATA / "test" / "test_images").glob("*.jpeg"))[:LIMIT]
assert paths, f"no test images in {DATA / 'test' / 'test_images'}"
t0 = time.perf_counter()
instances = pipe.run(paths, workers=4)  # 4 processes run the CPU stages while the GPU computes the maps
elapsed = time.perf_counter() - t0
n = submission.write(instances, OUT)
print(submission.validate(OUT, [p.stem for p in paths]))
print(f"{n} filaments in {len(paths)} frames at tau {pipe.tau}, {elapsed / len(paths):.2f} s per frame on {pipe.dev}")
print({k: round(v / len(paths), 3) for k, v in pipe.timings.items()})

## 4. A look at the result

The selected filaments of a few frames over their flat field.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from solarfil import preprocess, rle
from solarfil.data import read_image

fig, axes = plt.subplots(1, min(3, len(paths)), figsize=(18, 6), squeeze=False)
for ax, p in zip(axes[0], paths[:3]):
    flat, _ = preprocess.flat_field(read_image(p))
    ax.imshow(np.clip((flat - 1) * 3 + 0.5, 0, 1), cmap="gray")
    for counts in instances[p.stem]:
        ax.contour(rle.decode(counts), levels=[0.5], colors="red", linewidths=0.6)
    ax.set_title(p.stem)
    ax.axis("off")
plt.tight_layout()